In [1]:
import os
import json
import re

import pandas as pd

In [2]:
name_pattern = r'\n# [A-Za-z ]+\n'

In [3]:
# runs = [
#     os.path.join('experiments', str(x)) for x in range(744, 824)
# ] # without +/- fix in Z_0 calculation

runs = [
    os.path.join('experiments', str(x)) for x in range(826, 906)
] # fixed +/- in Z_0 calculation

assert len(runs) == 80, len(runs)

In [4]:
soln_number_to_name = {
    1: 'Plane waves',
    2: 'Radial waves',
    3: 'Hopf fibration',
    4: 'Random solution'
}

In [5]:
df = pd.DataFrame(
    columns=list(json.load(open(os.path.join(runs[0], 'hparams.json'))).keys())
    + [
        'soln_name',
        'min_mse_val',
        'time_to_min_mse_val',
        'min_rel_l2_error',
        'time_to_min_rel_l2_error',
        'time_to_5p_rel_l2_error',
        'time_to_1p_rel_l2_error'
    ]
)

for run in runs:
    info = json.load(open(os.path.join(run, 'hparams.json')))
    info['soln_name'] = soln_number_to_name[info['soln']]
    
    log = pd.read_csv(os.path.join(run, 'log.tsv'), sep='\t')
    
    # epoch where we reach minimum validation MSE
    min_mse_val = log['mse_val'].min()
    info['min_mse_val'] = min_mse_val
    
    min_mse_val_epoch = int(log[log['mse_val'] == min_mse_val].iloc[0]['epoch'])
    # how much training time to get there?
    info['time_to_min_mse_val'] = log[log['epoch'] <= min_mse_val_epoch]['training_time'].sum().item()
    
    # epoch where we reach minimum validation relative L2 error
    min_rel_l2_error = log['rel_l2_error'].min()
    info['min_rel_l2_error'] = min_rel_l2_error
    
    min_rel_l2_error_epoch = int(log[log['rel_l2_error'] == min_rel_l2_error].iloc[0]['epoch'])
    info['time_to_min_rel_l2_error'] = log[log['epoch'] <= min_rel_l2_error_epoch]['training_time'].sum().item()
    
    # epoch where we first reach 5% relative error
    first_5p_epoch = int(log[log['rel_l2_error'] < 0.05].iloc[0]['epoch'])
    info['time_to_5p_rel_l2_error'] = log[log['epoch'] <= first_5p_epoch]['training_time'].sum().item()

    first_1p_epoch = int(log[log['rel_l2_error'] < 0.01].iloc[0]['epoch'])
    info['time_to_1p_rel_l2_error'] = log[log['epoch'] <= first_1p_epoch]['training_time'].sum().item()
    
    df.loc[len(df)] = info

In [6]:
results = df[[
    'soln_name',
    'time_to_5p_rel_l2_error',
    'time_to_1p_rel_l2_error',
    'time_to_min_rel_l2_error',
    'min_rel_l2_error',
    'use_cpu',
    'add_bc',
    'seed'
]].rename(columns={
    'time_to_1p_rel_l2_error': 'Time to <1% RL2E. (s)',
    'time_to_5p_rel_l2_error': 'Time to <5% RL2E. (s)',
    'time_to_min_rel_l2_error': 'Time to min. RL2E. (s)',
    'min_rel_l2_error': 'Min. RL2E. (%)',
    'soln_name': 'Solution',
    'use_cpu': 'on CPU',
    'add_bc': 'w/ BC'
}).groupby(by=['Solution', 'on CPU', 'w/ BC']).agg(['mean', 'sem']).drop(columns=['seed'])

display_results = {}
metrics = results.columns.get_level_values(0).unique()

for col in metrics:
    mean = results[(col, 'mean')]
    sem = results[(col, 'sem')]
    if 'Time' in col:
        display_results[col] = mean.round(1).astype(str) + ' ± ' + sem.round(1).astype(str)
    else:  # relative error columns
        display_results[col] = mean.map('{:.2e}'.format) + ' ± ' + sem.map('{:.2e}'.format)

display_results = pd.DataFrame(display_results)
display_results

Time to <5% RL2E. (s) Time to <1% RL2E. (s)  \
Solution        on CPU w/ BC                                               
Hopf fibration  False  False             0.8 ± 0.0             2.4 ± 0.1   
                       True              2.2 ± 0.5             5.7 ± 2.2   
                True   False             0.8 ± 0.1             3.0 ± 0.1   
                       True              1.6 ± 0.1             3.6 ± 0.2   
Plane waves     False  False             3.1 ± 0.3             5.6 ± 0.3   
                       True              2.9 ± 0.1             8.8 ± 1.3   
                True   False             3.4 ± 0.3             6.1 ± 0.6   
                       True              3.1 ± 0.3             9.7 ± 0.7   
Radial waves    False  False             3.0 ± 0.2             8.3 ± 0.4   
                       True              4.3 ± 0.5            19.5 ± 3.8   
                True   False             2.9 ± 0.2             8.3 ± 0.4   
                       True              5.2 ± 0.5            29.6 ± 5.4   
Random solution False  False             1.9 ± 0.0             3.2 ± 0.1   
                       True              5.7 ± 1.2            38.7 ± 9.0   
                True   False             1.3 ± 0.0             2.3 ± 0.1   
                       True              6.2 ± 1.1            43.2 ± 6.2   

                             Time to min. RL2E. (s)       Min. RL2E. (%)  
Solution        on CPU w/ BC                                              
Hopf fibration  False  False            125.6 ± 2.4  3.09e-05 ± 2.44e-06  
                       True             114.9 ± 6.2  7.21e-04 ± 4.30e-04  
                True   False           191.5 ± 11.0  3.31e-05 ± 2.07e-06  
                       True             160.8 ± 6.4  2.89e-04 ± 1.40e-05  
Plane waves     False  False            115.8 ± 3.5  1.24e-04 ± 1.14e-05  
                       True             122.4 ± 5.2  1.00e-04 ± 1.84e-05  
                True   False           152.6 ± 12.8  1.17e-04 ± 1.09e-05  
                       True             144.2 ± 1.4  9.95e-05 ± 1.69e-05  
Radial waves    False  False            117.4 ± 5.1  4.05e-04 ± 1.53e-05  
                       True             121.4 ± 3.0  3.03e-03 ± 1.03e-03  
                True   False            135.1 ± 9.0  4.00e-04 ± 1.17e-05  
                       True             160.9 ± 5.3  3.59e-03 ± 7.66e-04  
Random solution False  False            116.5 ± 3.6  1.50e-04 ± 8.36e-06  
                       True             126.2 ± 4.3  2.52e-03 ± 6.70e-04  
                True   False            106.3 ± 1.0  1.49e-04 ± 6.22e-06  
                       True            128.4 ± 13.7  3.11e-03 ± 2.97e-04